In [5]:
# Standard library imports
import argparse
from enum import Enum
import json
import logging
import os
import sys
from textwrap import dedent as ded
from typing import Annotated, Any, Dict, List, NotRequired, Optional, TypedDict
from uuid import UUID
from datetime import datetime
from string import ascii_lowercase, ascii_uppercase

# Third-party imports
from IPython.display import Markdown, display
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessage, AnyMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import LLMResult
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import END, MessagesState, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
import pandas as pd
import pycountry
from pydantic import BaseModel, Field, conlist, constr, model_validator
import requests


# Variable assignments
dedent = lambda x: ded(x.strip())

In [7]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY") or os.getenv("TMDB_key")
TMDB_BASE = "https://api.themoviedb.org/3"

def get_global_streaming_providers(series_name: str):
    """
    Searches for a TV series on TMDB and lists ALL streaming providers 
    globally, categorized by country code.
    """
    if not TMDB_API_KEY:
        raise ValueError("TMDB API Key missing.")

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {TMDB_API_KEY}" if len(TMDB_API_KEY) > 50 else None
    }
    params = {"api_key": TMDB_API_KEY} if not headers["Authorization"] else {}

    # 1. Search for the TV series
    search_response = requests.get(f"{TMDB_BASE}/search/tv", headers=headers, params={**params, "query": series_name})
    search_response.raise_for_status()
    results = search_response.json().get("results", [])

    if not results:
        return f"No series found matching '{series_name}'."

    best_match = results[0]
    series_id = best_match["id"]
    official_name = best_match["name"]

    # 2. Get global watch providers
    providers_response = requests.get(f"{TMDB_BASE}/tv/{series_id}/watch/providers", headers=headers, params=params)
    providers_response.raise_for_status()
    global_results = providers_response.json().get("results", {})

    if not global_results:
        return f"No streaming provider data found anywhere globally for '{official_name}'."

    print(f"Global Streaming Profiles for: {official_name}\n{'='*40}")

    # 3. Iterate through every country payload TMDB returns
    for country, data in sorted(global_results.items()):
        # Skip the meta link item if present
        if not isinstance(data, dict):
            continue
            
        flatrate = [p["provider_name"] for p in data.get("flatrate", [])]
        free = [p["provider_name"] for p in data.get("free", [])]
        buy = [p["provider_name"] for p in data.get("buy", [])]
        
        # Only print countries that have actual actionable platforms attached
        if flatrate or free or buy:
            print(f"\n🌍 Country Code: {country}")
            if flatrate:
                print(f"  - Stream: {', '.join(flatrate)}")
            if free:
                print(f"  - Free: {', '.join(free)}")
            if buy:
                print(f"  - Rent/Buy: {', '.join(buy)}")



In [8]:
get_global_streaming_providers("Severance")

Global Streaming Profiles for: Severance

🌍 Country Code: AE
  - Stream: Apple TV

🌍 Country Code: AG
  - Stream: Apple TV

🌍 Country Code: AR
  - Stream: Apple TV

🌍 Country Code: AT
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Country Code: AU
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Country Code: AZ
  - Stream: Apple TV

🌍 Country Code: BE
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Country Code: BG
  - Stream: Apple TV

🌍 Country Code: BH
  - Stream: Apple TV

🌍 Country Code: BM
  - Stream: Apple TV

🌍 Country Code: BO
  - Stream: Apple TV

🌍 Country Code: BR
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Country Code: BS
  - Stream: Apple TV

🌍 Country Code: BY
  - Stream: Apple TV

🌍 Country Code: BZ
  - Stream: Apple TV

🌍 Country Code: CA
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Country Code: CH
  - Stream: Apple TV

🌍 Country Code: CL
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Country Code: CO
  - Stream: Apple TV, Apple TV Amazon Channel

🌍 Co